# ALPR — Colab Runner
**One-time setup:** Run cells 1–4 once per session.
**On code updates:** Just run Cell 2 (`git pull`) then Cell 5 to restart Streamlit.

> Make sure you have selected **Runtime → Change runtime type → T4 GPU** before starting.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ── Cell 2: Clone or pull latest code from GitHub ─────────────────
# First time:  clones the repo
# Every update: pulls latest changes

import os

REPO_URL = 'https://github.com/YOUR_USERNAME/ALPR.git'  # <-- change this
REPO_DIR = '/content/ALPR'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

print('Done — code is up to date')

In [ ]:
# ── Cell 3: Install dependencies (skip if already installed this session) ──
!pip install -q -r /content/ALPR/requirements.txt
print('Dependencies installed')

In [ ]:
# ── Cell 4: Install tunnel (Cloudflare — no bandwidth limit, no account needed) ──
!pip install -q pycloudflared

from pycloudflared import try_cloudflare
print('Cloudflare tunnel ready — will be started in Cell 5')

In [ ]:
# ── Cell 5: Launch FastAPI server + Cloudflare tunnel ─────────────
import subprocess, time, urllib.request, os, sys

# Kill any previous processes
os.system('pkill -f "server.py" 2>/dev/null; pkill -f gradio_app 2>/dev/null; pkill -f streamlit 2>/dev/null')
time.sleep(3)

# Install python-multipart if missing (required for file uploads)
os.system(f'{sys.executable} -m pip install -q python-multipart')

# Start FastAPI server in background
log_file = open('/content/server.log', 'w')
proc = subprocess.Popen(
    ['python', '-u', '/content/ALPR/server.py'],
    stdout=log_file, stderr=log_file
)

# Wait until server is accepting connections (up to 120s)
print('Waiting for server to start (loading models)...', end='', flush=True)
ready = False
for i in range(120):
    try:
        urllib.request.urlopen('http://localhost:8000/', timeout=2)
        ready = True
        print(f' ✅ ready ({i}s)')
        break
    except Exception:
        # Check if process died early
        if proc.poll() is not None:
            print(f'\n❌ Server process died (exit code {proc.returncode})')
            with open('/content/server.log') as f:
                print(f.read())
            raise RuntimeError('Server crashed on startup')
        if i % 10 == 9:
            print(f'[{i+1}s]', end='', flush=True)
        else:
            print('.', end='', flush=True)
        time.sleep(1)

if not ready:
    print('\n❌ Server timed out — last 50 lines of log:')
    with open('/content/server.log') as f:
        lines = f.readlines()
    print(''.join(lines[-50:]))
    raise RuntimeError('Server failed to start')

print('Server log (last 5 lines):')
with open('/content/server.log') as f:
    lines = f.readlines()
print(''.join(lines[-5:]))

# Open Cloudflare tunnel
from pycloudflared import try_cloudflare
result = try_cloudflare(port=8000, verbose=False)
tunnel_url = result.tunnel

print('\n' + '=' * 60)
print('  🌐 Your app is live at:')
print(f'  {tunnel_url}')
print('=' * 60)
print('\nIf you see Cloudflare Error 1033, just re-run this cell.')

In [ ]:
# ── Cell 6: Get a fresh tunnel URL (run this without restarting server) ──
from pycloudflared import try_cloudflare
result = try_cloudflare(port=8000, verbose=False)
print('=' * 60)
print('  🌐 Your app is live at:')
print(f'  {result.tunnel}')
print('=' * 60)

In [ ]:
# ── Cell 6 (optional): Upload a video to Colab for testing ────────
from google.colab import files
uploaded = files.upload()   # pick a video from your computer
for fname in uploaded:
    print(f'Uploaded: /content/{fname}')
    print('Use this path in the video app sidebar')

In [ ]:
# ── Cell 7 (optional): Mount Google Drive ─────────────────────────
# If your videos are already on Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted — files are at /content/drive/MyDrive/')